# Pittsburgh's Best Neighborhood — Accessibilty Sub-Metric

**Sub-metric:** Accessibility (number of transit stops per square mile)



### What this notebook does
1. Loads the City of Pittsburgh Regional Transit Stops dataset from the WPRDC.
2. Explores what's in the data.
3. Filters out data-entry outliers (incline).
4. Sums total number of stops per square mile.
5. Visualizes the ranking and the breakdown by neighborhood.
6. Normalizes the data to a 0–1 score so it combines into the group's metric nicely.
7. Names the most accessible neighborhood.

**Dataset source:** [Pittsburgh Regional Transit Stops — WPRDC](https://data.wprdc.org/dataset/prt-of-allegheny-county-transit-stops)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

stops = pd.read_csv('transit_stops.csv')

### Keep only Pittsburgh Neighborhoods
Removes rows were neighborhoods are missing

In [ ]:
stops = stops[stops["hood"].notna()].copy()

neighborhood = (
    stops.groupby("hood")
    .agg(
        unique_stops = ("stop_id", "nunique"),
        route_variety = ("routes", "nunique"),
        weekly_service = ("trips_7d", "sum")
    )
    .reset_index()
)

### Normalize Metrics
Compress stops into neighborhood-level metrics:
1) Number of unique stops
2) How many different bus routes serve the area/destinations reachable
3) Total number of weekly trips/service frequency

Normalize metrics by converting data into a 0-1 range, where 0 is lowest in the city and 1 is highest in the city

In [ ]:
def normalize(col):
    return (col - col.min()) / (col.max() - col.min())

neighborhood["stop_score"] = normalize(neighborhood["unique_stops"])
neighborhood["route_score"] = normalize(neighborhood["route_variety"])
neighborhood["service_score"] = normalize(neighborhood["weekly_service"])

### Weighted Accessibility Score
Combine the metrics into a single score, where each metric is given a different weight:
1) Number of stops (40%)
2) Bus routes (35%)
3) Service Frequency (25%)

Producing a single transit accessibility index for each neighborhood and then ranking from best to worst accessibility access

In [ ]:
neighborhood["accessibility_score"] = (
      neighborhood["stop_score"] * 0.40
    + neighborhood["route_score"] * 0.35
    + neighborhood["service_score"] * 0.25
).round(3)

neighborhood = neighborhood.sort_values(
    "accessibility_score",
    ascending=False
).reset_index(drop=True)

### Results
Creates a horizontal bar chart and a table displaying best-connected neighborhoods

In [ ]:
print(neighborhood.head(15))

top10 = neighborhood.head(10)

plt.barh(top10["hood"], top10["accessibility_score"])
plt.gca().invert_yaxis()
plt.title("Top Pittsburgh Neighborhoods by Transit Accessibility")
plt.xlabel("Accessibility Score")
plt.tight_layout()
plt.show()

### Saving Data

In [ ]:
neighborhood.to_csv("neighborhood_accessibility_scores.csv", index=False)
print("Saved neighborhood_accessibility_scores.csv")